[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Generative Models - Variational Auto Encoder - Celeb A

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 10/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0099DeepLearningObjectDetection.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F

import torchvision
from torchvision.transforms import v2 as TorchVisionTrns

import torchinfo
import torchvista

from torchmetrics.functional import r2_score

# Computer Vision
import imageio

# Miscellaneous
import math
import os
from platform import python_version
import random

# Typing
from typing import Callable, Literal, Optional, Self, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

TENSOR_BOARD_BASE = 'TB'

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataVisualization import PlotLabelsHistogram, PlotMnistImages, PlotScatterData
from DeepLearningPyTorch import TrainModelDataset

In [ ]:
# General Auxiliary Functions

def TensorToNumpy( tZ: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor to a Numpy Array.
    """
    tZ = tZ.squeeze()
    tX = tZ.detach().cpu().numpy()

    return tX

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor to a Numpy Array for image data.
    """
    tZ = TensorToNumpy(tZ)

    if tZ.ndim == 2:
        # Grayscale image
        tZ = np.expand_dims(tZ, axis = -1)
    elif tZ.ndim == 3 and tZ.shape[0] in [1, 3]:
        # Convert from (C, H, W) to (H, W, C)
        tZ = np.transpose(tZ, (1, 2, 0))
    
    return tZ #<! (H, W, C) format

class CelebADataset(torch.utils.data.Dataset):
    """
    CelebA Dataset from CSV File.
    Supports Image Classification and Self Supervised Learning tasks.
    """

    def __init__( self: Self, datasetPath: str, *, clsFileName: str = 'Classes.txt', hImgTrns: Optional[Callable] = None, hFeatTrns: Optional[Callable] = None, hTgtTrns: Optional[Callable] = None ) -> None:
        """
        Constructor Method.

        Parameters
        ----------
        datasetPath : str
            Path to the dataset files.
        subSetType : Literal['All', 'Train', 'Val']
            Subset type: 'All' for the entire dataset, 'Train' for training set, 'Val' for validation set
        hImgTrns : Optional[Callable], optional
            Transform to be applied to the images, by default None
        """

        # Folder contains files: 
        #   - Images: 000001.jpg, 000002.jpg, ...,
        #   - Labels: Classes.txt
        lFiles = sorted([f for f in os.listdir(datasetPath) if f.endswith('.jpg')])
        vLbl   = np.loadtxt(os.path.join(datasetPath, clsFileName), dtype = np.int64) #<! Zero based labels (Originally 1 based in the original dataset)

        # Assumption: All images have the same dimension
        # Read the 1st image to get the spatial dimensions
        tI = imageio.v3.imread(os.path.join(datasetPath, lFiles[0]))
        tuImgSize = (tI.shape[0], tI.shape[1])

        self._datasetPath = datasetPath
        self._numSamples  = len(lFiles)
        self._lFiles      = lFiles
        self._vLbl        = vLbl
        self._tuImgSize   = tuImgSize  #<! CelebA images are 218x178 RGB
        self._hImgTrns    = hImgTrns
        self._hTgtTrns    = hTgtTrns
        self._hFeatTrns   = hFeatTrns

    def __len__( self: Self ) -> int:
        """
        Returns the number of samples in the dataset.
        """
        return self._numSamples

    def __getitem__( self: Self, idx: int ) -> Tuple[Tensor, Tuple[Union[int, Tensor], Tensor]]:
        """
        Returns the sample at the given index.

        Parameters
        ----------
        idx : int
            Index of the sample to be retrieved.

        Returns
        -------
        Tuple[torch.Tensor, Tuple[Union[int, torch.Tensor], torch.Tensor]]
            Tuple containing the sample tensor and a tuple of the label and the target tensor.
        """

        imgPath = os.path.join(self._datasetPath, self._lFiles[idx])
        tX    = torchvision.io.decode_image(imgPath, 'RGB') #<! UInt8 Tensor
        valY  = self._vLbl[idx]

        if self._hImgTrns:
            # Assuming Torchvision v2 transforms
            tX = self._hImgTrns(tX)

        # Create a copy for feature transform
        # Handle the case of NumPy Array and Torch Tensor
        if isinstance(tX, np.ndarray):
            tY = tX.copy()
        else:
            tY = tX.clone()
        
        if self._hFeatTrns:
            tX = self._hFeatTrns(tX)
        
        if self._hTgtTrns:
            tY = self._hTgtTrns(tY)

        return tX, (valY, tY)
    
    def GetLabels( self: Self ) -> NDArray:
        """
        Returns all labels in the dataset.

        Returns
        -------
        NDArray
            Array of labels.
        """
        return self._vLbl

    def GetImageSpatialSize( self: Self ) -> Tuple[int, int]:
        """
        Returns the size of the images in the dataset.

        Returns
        -------
        Tuple[int, int]
            Tuple containing the height and width of the images.
        """
        return self._tuImgSize
    
    def SetTransform( self: Self, trnsType: Literal['Feature', 'Image', 'Target'], hTrns: Optional[Callable] ) -> None:
        """
        Sets the transform for the dataset.

        Parameters
        ----------
        trnsType : Literal['Feature', 'Image', 'Target']
            Type of transform to be set.
        hTrns : Optional[Callable]
            Transform to be applied.
        """

        match trnsType:
            case 'Feature':
                self._hFeatTrns = hTrns
            case 'Image':
                self._hImgTrns = hTrns
            case 'Target':
                self._hTgtTrns = hTrns
            case _:
                raise ValueError(f'Unsupported transform type: {trnsType}')

## Variational Auto Encoder

![](https://i.imgur.com/REzR5tu.png)
<!-- ![](https://i.postimg.cc/kg1bKpLg/Diagrams-Variational-Auto-Encoder.png) -->

A _Variational Auto Encoder_ (VAE) is a generative model which learns both a representation of the data and a structured latent space suitable for sampling.  
The model is composed of:
 - _Encoder_: Transforms the input $\color{cyan}{\boldsymbol{x}}$ into the parameters of a distribution: a mean $\boldsymbol{\mu} \left( {\color{cyan}{\boldsymbol{x}}} \right)$ and a vector of standard deviations $\boldsymbol{\sigma} \left( {\color{cyan}{\boldsymbol{x}}} \right)$.  
 - _Embedding_: Samples $\color{green}{\boldsymbol{z}}$ from the encoded distribution using the _Reparameterization Trick_: $\color{green}{\boldsymbol{z}} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$, where $\boldsymbol{\epsilon} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.  
 - _Decoder_: Reconstructs the input from $\color{green}{\boldsymbol{z}}$.  

 The VAE's _Prior_ encourages the encoded distributions to be close to $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.
 Basically each sample is a "blob" which makes nearby points in the latent space decode into plausible samples.

The VAE loss is composed of:
 - _Reconstruction Loss_  
   Encourages the decoded sample to match the input.
 - _Regularization Loss_  
   Uses the KL Divergence to make the encoded distribution close to the prior $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.

Some use cases of _Variational Auto Encoders_:

 - Data Generation  
   Sample $\color{green}{\boldsymbol{z}} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$ and pass it through the decoder.
 - Latent Space Interpolation  
   Moving smoothly in the latent space should generate smooth changes in the decoded samples.
 - Representation Learning  
   The latent variables provide a compact and structured representation of the data.

</br> 

* <font color='brown'>(**#**)</font> A regular _Auto Encoder_ may create gaps in its latent space, so an arbitrary latent sample may decode into an invalid image.  
  The VAE regularizes the latent space to reduce those gaps and make sampling meaningful.
* <font color='brown'>(**#**)</font> The reconstruction and regularization terms create a tradeoff between accurate reconstruction and a smooth latent space.

This notebook demonstrates:
 - Building an _Encoder_ which predicts $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$.
 - Applying the _Reparameterization Trick_ to enable gradient based training.
 - Building a _Decoder_ based on CNN.
 - Training the VAE using reconstruction and KL Divergence losses.
 - Sampling from the prior to generate new images.
 - Exploring the geometry of the latent space.

</br>

* <font color='brown'>(**#**)</font> More expressive generative models, such as GANs and Diffusion Models, may produce "sharper" images, while VAEs offer a simple and explicit probabilistic latent space.

In [ ]:
# Parameters

# Data
folderName = 'CelebAAligned'
folderPath = os.path.join(DATA_FOLDER_PATH, folderName)

# Model
modelName = 'ModelVAECelebA_2026_08_29.pt' #<! OneDrive -> Courses -> Models -> AIProgram
latDim    = 32

# Loss
recLossType = 'MSE'
β           = 1.0

# Training
batchSize  = 16
numWorkers = 2 #<! Number of workers
numEpochs  = 45

# Visualization
numImg = 3

## Generate / Load Data

### Celeb A Dataset

The [CelebA dataset](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) contains more than 200,000 celebrity face images with identity labels and 40 binary facial attributes.  
In its aligned variant, facial landmarks are used to normalize the position, scale, and orientation of each face before cropping.  
This reduces irrelevant geometric variation, allowing the VAE to focus its latent representation on appearance properties such as expression, hair, and facial features.

In [ ]:
# Data Set

dsTrain = CelebADataset(folderPath)
vLbl    = dsTrain.GetLabels()

numSamples = len(dsTrain)
tuImgSize  = dsTrain.GetImageSpatialSize()

print(f'The number of samples in the data set: {numSamples:,}')
print(f'The number of classes in the data set: {len(np.unique(vLbl)):,}')

In [ ]:
# Element of the Data Set / Data Sample

tX, tuY = dsTrain[0]
valY    = tuY[0]
tY      = tuY[1]

print(f'The features shape: {tX.shape}')
print(f'The target shape  : {tY.shape}')
print(f'The label         : {valY}')

### Plot the Data

In [ ]:
# Plot the Data

# Plot a Grid of (`numImg`, `numImg`) random images from the data set
# Assuming numImg is defined elsewhere
hF, vHa = plt.subplots(ncols = numImg, nrows = numImg)
vHa = vHa.flat

for ii, hA in enumerate(vHa):
    idx = np.random.randint(0, numSamples)
    tX, (valY, _) = dsTrain[idx]
    
    tI = TensorImageNumpy(tX)
    
    hA.imshow(tI)
    hA.set_title(f'Label: {valY}')
    hA.axis('off')

In [ ]:
# Plot Single Sample

randIdx = random.randint(0, len(dsTrain) - 1)
tX, (valY, _) = dsTrain[randIdx]

tX = TensorImageNumpy(tX)

hF, hA = plt.subplots(figsize = (7, 7))
hA.imshow(tX)
hA.set_title(f'Sample Index: {randIdx}, Label: {valY}');

In [ ]:
# Plot Images of the Same Class
# Plot (`numImg`, `numImg`) random images of the same class from the data set.

classLbl = random.choice(np.unique(vLbl))
classLbl = 3953
vIDx = np.where(vLbl == classLbl)[0]

hF, vHa = plt.subplots(ncols = numImg, nrows = numImg)
vHa = vHa.flat

for ii, hA in enumerate(vHa):
    idx = np.random.choice(vIDx)
    tX, (valY, _) = dsTrain[idx]
    
    tI = TensorImageNumpy(tX)
    
    hA.imshow(tI)
    hA.set_title(f'Label: {valY}')
    hA.axis('off')

In [ ]:
# Histogram of Labels

# Takes time!!!
# hF, hA = plt.subplots(figsize = (8, 4))
# hA = PlotLabelsHistogram(dsTrain.GetLabels(), hA = hA)
# hA.set_title('Histogram of Labels');

### Augmentation / Transform

In [ ]:
# Loader Transform

oTrns = TorchVisionTrns.Compose([
    TorchVisionTrns.RandomHorizontalFlip(p = 0.5),
    TorchVisionTrns.CenterCrop(148),
    TorchVisionTrns.Resize(64),
    TorchVisionTrns.ToDtype(torch.float, scale = True),
])

In [ ]:
# Apply Transforms

dsTrain.SetTransform('Image', oTrns)

In [ ]:
# Element of the Data Set / Data Sample

tX, tuY = dsTrain[0]
valY    = tuY[0]
tY      = tuY[1]

print(f'The features type        : {type(tX)}')
print(f'The features element type: {tX.dtype}')
print(f'The features shape       : {tX.shape}')
print(f'The target type          : {type(tY)}')
print(f'The target element type  : {tY.dtype}')
print(f'The target shape         : {tY.shape}')
print(f'The label type           : {type(valY)}')
print(f'The label                : {valY}')

In [ ]:
# Plot Single Sample

randIdx = random.randint(0, len(dsTrain) - 1)
tX, tuY = dsTrain[randIdx]
valY    = tuY[0]

mX = TensorImageNumpy(tX)

hF, hA = plt.subplots(figsize = (4, 4))
hA.imshow(mX, cmap = 'gray')
hA.set_title(f'Sample Index: {randIdx}, Label: {valY}');

### Data Loaders

In [ ]:
# Data Loader

dlData = torch.utils.data.DataLoader(dsTrain, batch_size = batchSize, shuffle = True, num_workers = 0, pin_memory = torch.cuda.is_available())

In [ ]:
# Iterate on the Loader
# The first batch.
tX, tuY = next(iter(dlData)) #<! PyTorch Tensors

print(f'The batch features dimensions: {tX.shape}')
print(f'The batch labels dimensions: {tuY[0].shape}')
print(f'The batch targets dimensions: {tuY[1].shape}')

## Build Variational Auto Encoder Model

In a _Variational Auto Encoder_ the encoder does not map each sample into a single point.  
It maps the sample ${\color{cyan}{\boldsymbol{x}}}_{i}$ into a distribution in the latent space:

$$ q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i} \right) = \mathcal{N} \left( \boldsymbol{\mu}_{i}, \operatorname{Diag} \left( \boldsymbol{\sigma}_{i}^{2} \right) \right) $$

A latent vector is sampled using the _Reparameterization Trick_:

$$ {\color{green}{\boldsymbol{z}}}_{i} = \boldsymbol{\mu}_{i} + \boldsymbol{\sigma}_{i} \odot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right) $$

This form separates the random sampling from $\boldsymbol{\mu}_{i}$ and $\boldsymbol{\sigma}_{i}$, hence gradients can propagate through the encoder.

The VAE objective can be written as:

$$ \arg \min_{\boldsymbol{w}} \sum_{i} \left[ \mathbb{E}_{q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i} \right)} \left[ \mathcal{L}_{\mathrm{Rec}} \left( {\phi}_{\boldsymbol{w}} \left( \boldsymbol{z} \right), {\color{cyan}{\boldsymbol{x}}}_{i} \right) \right] + \beta D_{\mathrm{KL}} \left( q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i} \right) \,\|\, \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right) \right) \right] $$

Where ${\phi}_{\boldsymbol{w}}$ is the decoder and $\beta$ controls the regularization strength.

The objective is composed of:
 - _Reconstruction Loss_  
   Makes the decoded sample similar to the input.  
   In case of Gaussian Noise it matches: $\frac{1}{N} \sum_{i = 1}^{N} {\left\| {\color{cyan}{\boldsymbol{x}}}_{i} - {\phi}_{\boldsymbol{w}} \left( {\color{green}{\boldsymbol{z}}}_{i}\right) \right\|}_{2}^{2}$
 - _KL Divergence Loss_  
   Makes each encoded distribution similar to the prior $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.  
   This organizes the latent space and makes sampling from the prior meaningful.  
   In the case above, for a diagonal covariance, it sums to: $D_{\mathrm{KL}}\left(\mathcal{N}_{d}\left(\boldsymbol{\mu},\text{diag}\left(\boldsymbol{\sigma}\right)\right)||\mathcal{N}_{d}\left(\boldsymbol{0},\boldsymbol{I}\right)\right)=\frac{1}{2}\sum_{i=1}^{d}\left(\sigma_{i}^{2}+\mu_{i}^{2}-1-\log\left(\sigma_{i}^{2}\right)\right)$.

Hence the overall Loss Function is given by:

$$ \mathcal{L}_{\mathrm{VAE}} = \frac{1}{N} \sum_{i = 1}^{N} \left[ {\left\| {\color{cyan}{\boldsymbol{x}}}_{i} - {\phi}_{\boldsymbol{w}} \left( {\color{green}{\boldsymbol{z}}}_{i} \right) \right\|}_{2}^{2} + \frac{\beta}{2} \sum_{j = 1}^{d} \left( \sigma_{i,j}^{2} + \mu_{i,j}^{2} - 1 - \log \left( \sigma_{i,j}^{2} \right) \right) \right] $$

</br>

* <font color='brown'>(**#**)</font> A small $\beta$ favors accurate reconstruction but may create a less organized latent space.  
  At the extreme $\beta = 0$, it is equivalent to _Auto Encoder_.
* <font color='brown'>(**#**)</font> A large $\beta$ creates stronger regularization but may remove useful information from the latent representation.  
* <font color='brown'>(**#**)</font> During training, the expectation is usually approximated using a single sample of $\boldsymbol{\epsilon}$ for each input.
* <font color='brown'>(**#**)</font> The encoder commonly predicts $\log \left( \boldsymbol{\sigma}^{2} \right)$ instead of $\boldsymbol{\sigma}$ for numerical stability.

In [ ]:
# Sampling Layer

class GaussianSamplingLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, mμ: Tensor, mLogΣ: Tensor) -> Tensor:
        """
        Args:
            mμ   : Tensor of mean values (batchSize, latDim)
            mLogΣ: Tensor of log variances (batchSize, latDim)
        
        Note: Using `mLogΣ` instead of `mσ` for numerical stability.
        """

        mσ = torch.exp(0.5 * mLogΣ) #<! The standard deviation from log variance
        mε = torch.randn_like(mσ) #<! Sample random noise from standard normal distribution
        mZ = mμ + mε * mσ #<! Reparameterization trick: z = μ + σ * ε, where ε ~ N(0, I)

        return mZ

In [ ]:
# Building Blocks

# Inverted Residual Block

class InvertedResidualBlock(nn.Module):
    """
    Modern block structure: Expand -> Depth Wise Convolution -> Project.
    Includes a skip connection if input and output shapes match.
    """
    def __init__(self, numChnlIn: int, numChnlOut: int, expFctr: int = 4, strideSize: int = 1):
        super().__init__()
        
        self.strideSize = strideSize
        self.enableSkip = (strideSize == 1 and numChnlIn == numChnlOut)
        hiddenDim       = numChnlIn * expFctr

        self.oBlock = nn.Sequential(
            # Expansion of Channels (Using 1x1 Convolution)
            nn.Conv2d(numChnlIn, hiddenDim, 1, bias = False),
            nn.BatchNorm2d(hiddenDim),
            nn.SiLU(),
            
            # Depth Wise Convolution (3x3)
            nn.Conv2d(hiddenDim, hiddenDim, 3, stride = strideSize, padding = 1, groups = hiddenDim, bias = False),
            nn.BatchNorm2d(hiddenDim),
            nn.SiLU(),
            
            # Projection (Using 1x1 Convolution) - Linear bottleneck (No activation at end)
            nn.Conv2d(hiddenDim, numChnlOut, 1, bias = False),
            nn.BatchNorm2d(numChnlOut),
        )

    def forward(self, tX: Tensor) -> Tensor:
        
        if self.enableSkip:
            return tX + self.oBlock(tX)
        else:
            return self.oBlock(tX)

# Depthwise Separable Convolution Block

class DepthwiseSeparableConv(nn.Module):
    """
    The building block of efficient networks.
    Splits a standard convolution into:
    1. Depthwise: Spatial filtering (lightweight)
    2. Pointwise: Channel mixing (1x1 conv)
    """
    def __init__(self, numChnlIn: int, numChnlOut: int, strideSize: int = 1):
        super().__init__()
        
        self.strideSize = strideSize
        
        # Depthwise Convolution (3x3) (Each kernel per channel)
        self.oBlock001 = nn.Sequential(
            nn.Conv2d(numChnlIn, numChnlIn, kernel_size = 3, padding = 1,  stride = strideSize, groups = numChnlIn, bias = False),
            nn.BatchNorm2d(numChnlIn),
            nn.SiLU(), #<! Modern activation (Swish)
        )
        # Pointwise Convolution (1x1) Projection over channels
        self.oBlock002 = nn.Sequential(
            nn.Conv2d(numChnlIn, numChnlOut, kernel_size = 1, bias = False),
            nn.BatchNorm2d(numChnlOut),
            nn.SiLU(), #<! Modern activation (Swish)
        )

    def forward(self, tX: Tensor) -> Tensor:
        
        tX = self.oBlock001(tX)
        tX = self.oBlock002(tX)
        
        return tX

In [ ]:
# Encoder Decoder Model
class DecoderStage(nn.Module):
    def __init__(self, numChnlIn: int, numChnlOut: int, tuOutSize: Tuple[int, int]) -> None:
        super().__init__()
        self.oStage = nn.Sequential(
            nn.Upsample(size = tuOutSize, mode = 'bilinear', align_corners = False),
            DepthwiseSeparableConv(numChnlIn, numChnlOut),
            InvertedResidualBlock(numChnlOut, numChnlOut, expFctr = 2),
        )

    def forward(self, tX: Tensor) -> Tensor:
        return self.oStage(tX)

class VariationalAutoEncoder(nn.Module):
    def __init__(self, latDim: int) -> None:
        super().__init__()

        self.oEncFeatures = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 5, stride = 2, padding = 2, bias = False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            InvertedResidualBlock(32, 48, strideSize = 2),
            InvertedResidualBlock(48, 64, strideSize = 2),
            InvertedResidualBlock(64, 96, strideSize = 2),
            InvertedResidualBlock(96, 128, strideSize = 2),
            InvertedResidualBlock(128, 160),
            InvertedResidualBlock(160, 160),
        )

        self.oEncHead = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(160, 2 * latDim),
        )

        self.oSampler = GaussianSamplingLayer()

        self.oDecInput = nn.Sequential(
            nn.Linear(latDim, 160 * 2 * 2),
            nn.SiLU(),
            nn.Unflatten(dim = 1, unflattened_size = (160, 2, 2)),
        )

        self.oDec = nn.Sequential(
            InvertedResidualBlock(160, 160, expFctr = 2),
            DecoderStage(160, 128, (4, 4)),
            DecoderStage(128, 96, (8, 8)),
            DecoderStage(96, 64, (16, 16)),
            DecoderStage(64, 48, (32, 32)),
            DecoderStage(48, 32, (64, 64)),
            nn.Conv2d(32, 3, kernel_size = 3, padding = 1),
            nn.Sigmoid(),
        )

    def forward(self, tX: Tensor) -> Tuple[Tensor, Tensor, Tensor]:

        mμ, mLogΣ = self.oEncHead(self.oEncFeatures(tX)).chunk(2, dim = 1)
        if self.training:
            mZ = self.oSampler(mμ, mLogΣ)
        else:
            mZ = mμ
        tXHat = self.oDec(self.oDecInput(mZ))

        return tXHat, mμ, mLogΣ

    def GetEmbedding(self, tX: Tensor) -> Tensor:

        mμ, _ = self.oEncHead(self.oEncFeatures(tX)).chunk(2, dim = 1)

        return mμ

In [ ]:
# The Model Object

oModel = VariationalAutoEncoder(latDim)

In [ ]:
# Model Summary

torchinfo.summary(oModel, (1, *tX.shape), col_names = ['kernel_size', 'input_size', 'output_size', 'num_params'], device = 'cpu', row_settings = ['depth', 'var_names'])

In [ ]:
# Model Graph

torchvista.trace_model(oModel.eval(), torch.unsqueeze(tX, 0))

## Train the Model

This section defines:
 - Loss Class (_VAE Loss_) 
   Composed of 2 loses:
   - The Reconstruction Loss  
     Based on the [MSE](https://en.wikipedia.org/wiki/Mean_squared_error) or [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) loss.  
     Drives reconstruction of the image at output.
   - The Prior Loss
     Based on the [Kullback Leibler Divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) Loss.  
     Promotes global structure of the latent space to match Normal Distribution.
 - Score  
   The R2 score.

In [ ]:
# Check GPU Availability

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device
print(f'The run device: {runDevice}')

In [ ]:
# The Loss Class
class VariationalAutoEncoderLoss(nn.Module):
    def __init__( self, recLossType: Literal['MAE', 'MSE'], β: float ) -> None:
        super().__init__()

        # The KL Loss is averaged over the batch size.
        # To keep scaling similar, the reconstruction loss is summed over all elements and averaged with the KL Loss.
        match recLossType:
            case 'MAE':
                self.oRecLoss = nn.L1Loss(reduction = 'sum')
            case 'MSE':
                self.oRecLoss = nn.MSELoss(reduction = 'sum')
            case _:
                raise ValueError(f'Unsupported loss type: {recLossType}')
        
        self.β = β
    
    def forward( self, tuYHat: Tuple[Tensor, Tensor, Tensor], tuY: Tuple[Tensor, Tensor] ) -> Tensor:

        tXHat, mμ, mLogΣ = tuYHat
        _,      tX        = tuY

        batchSize = tX.size(0)
        
        recLoss = self.oRecLoss(tXHat, tX)
        klLoss  = 0.5 * torch.sum(mLogΣ.exp() + mμ.square() - 1.0 - mLogΣ)

        return (recLoss + self.β * klLoss) / batchSize

In [ ]:
# The Score Class
class AutoEncoderScore(nn.Module):
    def __init__( self ) -> None:
        super().__init__()
    
    def forward( self, tuYHat: Tuple[Tensor, Tensor], tuY: Tuple[Tensor, Tensor] ) -> Tensor:

        tXHat, _, _ = tuYHat
        _, tX       = tuY
        
        r2Score = r2_score(tXHat.view(-1), tX.view(-1))
        
        return r2Score

In [ ]:
# Loss and Score
hL = VariationalAutoEncoderLoss(recLossType, β)
hS = AutoEncoderScore()
hL = hL.to(runDevice) #<! Not required!
hS = hS.to(runDevice)

In [ ]:
# Training the Model
oModel = oModel.to(runDevice) #<! Transfer model to device
oOpt = torch.optim.AdamW(oModel.parameters(), lr = 6e-4, betas = (0.9, 0.99), weight_decay = 1e-3) #<! Define optimizer
oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = 2e-3, total_steps = numEpochs)
oModel, lTrainLoss, lTrainScore, lValLoss, lValScore, lLearnRate = TrainModelDataset(oModel, dlData, oOpt, numEpochs, hL, hS, oSch = oSch)

In [ ]:
# Load the Model
modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)
if os.path.isfile(modelPath):
    dModel = torch.load(modelPath, map_location = runDevice)
    oModel.load_state_dict(dModel['Model'])
    print(f'Model loaded from: {modelPath}')
    oModel = oModel.to(runDevice) #<! Transfer model to device

In [ ]:
# Plot Training Phase
# Requires the variables: `lTrainLoss`, `lTrainScore`, `lValLoss`, `lValScore`, `lLearnRate` from training phase.

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (15, 5))
vHa = np.ravel(vHa)

hA = vHa[0]
hA.plot(lTrainLoss, lw = 2, label = 'Train')
hA.plot(lValLoss, lw = 2, label = 'Validation')
hA.set_title(f'VAE Loss ({recLossType}, β = {β})')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lTrainScore, lw = 2, label = 'Train')
hA.plot(lValScore, lw = 2, label = 'Validation')
hA.set_title('VAE Reconstruction Score')
hA.set_xlabel('Epoch')
hA.set_ylabel('Score')
hA.legend()

hA = vHa[2]
hA.plot(lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

In [ ]:
# Inference Mode

oModel = oModel.eval()

In [ ]:
# Sample from Train
tX, (_, tY) = dsTrain[7]

tX = tX.to(runDevice).unsqueeze(0)

with torch.inference_mode():
    tXHat, _, _ = oModel(tX)

mXHat = TensorImageNumpy(tXHat)
mX    = TensorImageNumpy(tX)

hF, vHa = plt.subplots(nrows = 1, ncols = 2, figsize = (6, 3))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(mX, cmap = 'gray')
hA.set_title('Input Image')
hA = vHa[1]
hA.imshow(mXHat, cmap = 'gray')
hA.set_title('Reconstructed Image');

In [ ]:
# Grid of Random Samples from the Dataset

hF, vHa = plt.subplots(nrows = 10, ncols = 10, figsize = (8, 8))

for ii, hA in zip(np.random.choice(numSamples, 100, replace = False), vHa.flat):
    tX, _ = dsTrain[ii]
    hA.imshow(TensorImageNumpy(tX))
    hA.axis('off')

hF.tight_layout(pad = 0.1);

In [ ]:
# Grid of Random Samples from the Latent Space

mZ = torch.randn(100, latDim, device = runDevice)

with torch.inference_mode():
    tXHat = oModel.oDec(oModel.oDecInput(mZ))

hF, vHa = plt.subplots(nrows = 10, ncols = 10, figsize = (8, 8))

for tX, hA in zip(tXHat, vHa.flat):
    hA.imshow(TensorImageNumpy(tX))
    hA.axis('off')

hF.tight_layout(pad = 0.1);

* <font color='red'>(**?**)</font> Explain the smoothness of the results of the model compared to the dataset.

In [ ]:
# Analysis of the Latent Space

numEncSamples = 2_000
lZ = []

with torch.inference_mode():
    for tX, _ in dlData:
        lZ.append(oModel.GetEmbedding(tX.to(runDevice)).cpu())
        if sum(mZ.shape[0] for mZ in lZ) >= numEncSamples:
            break

mZData = torch.cat(lZ)[:numEncSamples]
vZMean = mZData.mean(dim = 0)
_, _, mBasis = torch.pca_lowrank(mZData, q = 2, center = True)
mZPca = (mZData - vZMean) @ mBasis

# Project samples from the VAE prior onto the same PCA plane
mZPrior = torch.randn_like(mZData)
mPriorPca = (mZPrior - vZMean) @ mBasis

hF = plt.figure(figsize = (12, 7))
oGrid = hF.add_gridspec(2, 9, height_ratios = (3, 1))

hA = hF.add_subplot(oGrid[0, :])
hA.scatter(mPriorPca[:, 0], mPriorPca[:, 1], s = 8, alpha = 0.25, label = 'Prior samples')
hA.scatter(mZPca[:, 0], mZPca[:, 1], s = 8, alpha = 0.35, label = 'Encoded images')
hA.set_title('Encoded Images Compared with the Normal Prior')
hA.set_xlabel('Principal Component 1')
hA.set_ylabel('Principal Component 2')
hA.axis('equal')
hA.legend()

# Decode points along the dominant direction of the encoded data
vStep      = torch.linspace(-2.5, 2.5, 9)
valScale   = mZPca[:, 0].std()
mZTraverse = vZMean + vStep[:, None] * valScale * mBasis[:, 0]

with torch.inference_mode():
    tXHat = oModel.oDec(oModel.oDecInput(mZTraverse.to(runDevice)))

for ii, (tX, valStep) in enumerate(zip(tXHat, vStep)):
    hA = hF.add_subplot(oGrid[1, ii])
    hA.imshow(TensorImageNumpy(tX))
    hA.set_title(f'{valStep:.1f}σ')
    hA.axis('off')

hF.suptitle('Latent Space Structure and Traversal')
hF.tight_layout();